# 03 · Transformers for Sensor Time Series
### Community environmental monitoring, beyond text and images

Transformers are not only for language and images. The same "attention over a
sequence" idea works on **sensor data over time** — and that is increasingly how
communities monitor their own environment: low-cost **air-quality** sensors,
water and soil probes, weather stations, and acoustic monitors for wildlife.

In this notebook we simulate a **community air-quality station** (temperature,
humidity, and PM2.5 particulate matter) and build models that decide whether a
time window contains a **pollution event**. Along the way:

> **Who owns the sensor data? Who decides what counts as "normal" vs. an
> "anomaly"? And how do we keep monitoring from sliding into surveillance?**

We generate the data synthetically so the notebook always runs without downloads —
and so you can *see exactly* how the labels were defined.


> ✅ **Solution / runnable reference version.** This is the fully-solved version
> of the workshop notebook: every *Your turn* blank is filled in and code comments
> are trimmed, so you can **Runtime → Run all** and confirm it works end to end.
> The teaching version (with blanks to complete) lives in the parent `workshop3/` folder.


## How to use this notebook

This is **Workshop 3** of the *Introduction to AI* series. Workshops 1 and 2
looked at tabular data, images, tokenization and retrieval. Workshop 3 is a
hands-on tour of the **Hugging Face** ecosystem across three applications.

The notebooks are designed to be run in **Google Colab**, top to bottom.

Each notebook is organised in three levels:

- 🟢 **Beginner** — run pretrained models with a few lines of code.
- 🟡 **Medium** — adapt models, change parameters, and read the results critically.
- 🔴 **Optional / Advanced** — train or fine-tune a model yourself (best with a GPU).

Look out for these blocks:

- `# TODO:` cells are **your turn to code**. Try them before scrolling to the solution.
- **▶️ Play with it** cells are safe to edit and rerun — change models, labels, or numbers.
- **🧭 Critical reflection** boxes connect the technical work to the bigger questions of
  the workshop: *who designs these systems, who do they serve, whose knowledge do they
  include, and whose values do they encode?*

> 💡 **Before you start:** switch on a GPU via
> `Runtime -> Change runtime type -> Hardware accelerator -> GPU`.
> Everything still runs on CPU, just more slowly.

Workshop 3 sequence:
1. `01_computer_vision_with_transformers.ipynb`
2. `02_finetuning_transformers_for_text.ipynb`
3. `03_transformers_for_sensor_time_series.ipynb`


### 🧭 Why community sensing is political, not just technical

- **Data sovereignty:** when a community runs its own sensors, it controls the
  evidence about its own air, water and land. That can shift power in disputes with
  polluters or authorities.
- **Defining "normal":** an "anomaly" threshold is a *value judgement*. A level that
  is "acceptable" to a regulator may be unacceptable to the people breathing it.
- **Dual use:** the same time-series models that watch the environment can watch
  *people* (movement, occupancy, behaviour). Keep asking who is being monitored.


In [ ]:
import random
import sys
from pathlib import Path

import numpy as np

try:
    import torch
except ImportError:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

PIPELINE_DEVICE = 0 if DEVICE == "cuda" else -1

print("Python version:", sys.version.split()[0])
print("Detected device:", DEVICE)
if DEVICE == "cpu":
    print("Tip: enable a GPU via Runtime -> Change runtime type for faster runs.")
print("Seed set to:", SEED)

NOTEBOOK_FRAMING = "transformers for community environmental sensor monitoring"
NOTEBOOK_FRAMING


In [ ]:
!pip -q install scikit-learn matplotlib

print("Sensor time-series notebook ready.")


## 🟢 Part 1 — Build a synthetic sensor dataset

Each **sample** is a short window of readings from 3 sensors:
`temperature`, `humidity`, `PM2.5`. We create two classes:

- **class 0 — normal:** PM2.5 stays low and noisy.
- **class 1 — pollution event:** a bump in PM2.5 (e.g. smoke, traffic, a fire)
  while humidity dips.

Because *we* wrote the generator, we can see precisely how "event" was defined —
in the real world that definition is a contested choice.


In [ ]:
import numpy as np

SEQ_LEN = 48
CHANNELS = ["temperature", "humidity", "pm25"]
NUM_CHANNELS = len(CHANNELS)

def make_sensor_dataset(n_per_class=400, seq_len=SEQ_LEN, seed=0):
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 1, seq_len)
    X, y = [], []

    for _ in range(n_per_class):

        temp = 22 + 3 * np.sin(2 * np.pi * t + rng.uniform(0, 6.28)) + rng.normal(0, 0.3, seq_len)
        hum = 55 + 8 * np.sin(2 * np.pi * t + rng.uniform(0, 6.28)) + rng.normal(0, 1.0, seq_len)
        pm = np.clip(12 + rng.normal(0, 2, seq_len), 0, None)
        X.append(np.stack([temp, hum, pm], axis=-1))
        y.append(0)

    for _ in range(n_per_class):

        temp = 22 + 3 * np.sin(2 * np.pi * t + rng.uniform(0, 6.28)) + rng.normal(0, 0.3, seq_len)
        hum = 50 + 8 * np.sin(2 * np.pi * t + rng.uniform(0, 6.28)) + rng.normal(0, 1.0, seq_len)
        center = rng.integers(seq_len // 4, 3 * seq_len // 4)
        width = rng.uniform(2, 5)
        amplitude = rng.uniform(25, 60)
        bump = amplitude * np.exp(-0.5 * ((np.arange(seq_len) - center) / width) ** 2)
        pm = np.clip(12 + bump + rng.normal(0, 2, seq_len), 0, None)
        X.append(np.stack([temp, hum, pm], axis=-1))
        y.append(1)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    perm = rng.permutation(len(X))
    return X[perm], y[perm]

X, y = make_sensor_dataset(seed=SEED)
print("X shape:", X.shape, "(samples, timesteps, channels)")
print("y shape:", y.shape, "| class balance:", np.bincount(y))


In [ ]:
import matplotlib.pyplot as plt

normal_idx = np.where(y == 0)[0][0]
event_idx = np.where(y == 1)[0][0]

fig, axes = plt.subplots(1, NUM_CHANNELS, figsize=(15, 4))
for c, name in enumerate(CHANNELS):
    axes[c].plot(X[normal_idx, :, c], label="normal", marker=".")
    axes[c].plot(X[event_idx, :, c], label="event", marker=".")
    axes[c].set_title(name)
    axes[c].legend()
plt.suptitle("Normal vs. pollution-event window per sensor")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

mean = X_train.reshape(-1, NUM_CHANNELS).mean(axis=0)
std = X_train.reshape(-1, NUM_CHANNELS).std(axis=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std
print("Standardised. Train:", X_train.shape, "Test:", X_test.shape)


## 🟢 Part 2 — A simple, honest baseline

Before reaching for a transformer, build a baseline. We summarise each window with
a few statistics per channel (mean, std, max) and train a logistic regression.
A strong baseline keeps us honest about whether a fancy model is actually needed.


In [ ]:
def extract_features(windows):
    means = windows.mean(axis=1)
    stds = windows.std(axis=1)
    maxs = windows.max(axis=1)
    return np.concatenate([means, stds, maxs], axis=1)

F_train = extract_features(X_train)
F_test = extract_features(X_test)
print("Feature shape:", F_train.shape)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

baseline = LogisticRegression(max_iter=1000)
baseline.fit(F_train, y_train)
base_pred = baseline.predict(F_test)

print(f"Baseline accuracy: {accuracy_score(y_test, base_pred):.3f}")
print(f"Baseline macro-F1: {f1_score(y_test, base_pred, average='macro'):.3f}")


## 🟡 Part 3 — A Transformer encoder for sequences

Now the main event. A transformer treats the 48 timesteps like a sequence of
"tokens" and uses **self-attention** to let every timestep look at every other one.
The recipe:

1. **Project** each timestep's 3 sensor values into a `d_model`-dim vector.
2. Add **positional encoding** (attention has no built-in sense of order).
3. Pass through a stack of **TransformerEncoder** layers.
4. **Pool** across time and classify.


In [ ]:
import math
import torch
from torch import nn

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


In [ ]:
class SensorTransformer(nn.Module):

    def __init__(self, num_channels, num_classes=2, d_model=64, nhead=4, num_layers=2,
                 seq_len=SEQ_LEN, use_pos_encoding=True):
        super().__init__()
        self.use_pos_encoding = use_pos_encoding
        self.input_proj = nn.Linear(num_channels, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=0.1, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        h = self.input_proj(x)
        if self.use_pos_encoding:
            h = self.pos_encoder(h)
        h = self.encoder(h)
        h = h.mean(dim=1)
        return self.head(h)

_test = SensorTransformer(NUM_CHANNELS)
print(_test(torch.randn(2, SEQ_LEN, NUM_CHANNELS)).shape)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_loader(features, labels, batch_size=32, shuffle=False):
    ds = TensorDataset(torch.tensor(features), torch.tensor(labels))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def train_sensor_model(model, epochs=15, lr=1e-3, batch_size=32):
    model = model.to(DEVICE)
    train_loader = make_loader(X_train, y_train, batch_size, shuffle=True)
    test_loader = make_loader(X_test, y_test, batch_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb).argmax(-1)
                correct += (pred == yb).sum().item()
                total += len(yb)
        if (epoch + 1) % 3 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1:>2}/{epochs}  test accuracy: {correct / total:.3f}")
    return model


In [ ]:
torch.manual_seed(SEED)
model = SensorTransformer(NUM_CHANNELS, d_model=64, nhead=4, num_layers=2)
model = train_sensor_model(model, epochs=15, lr=1e-3)


### ▶️ Play with it — change the architecture

Rerun the cell below with different settings and compare test accuracy *and*
training time. Try:

- `d_model`: 16, 32, 64, 128
- `nhead`: 1, 2, 4, 8  (must divide `d_model`)
- `num_layers`: 1, 2, 4
- in `train_sensor_model`: different `lr` (1e-2, 1e-3, 1e-4) and `epochs`

Does the transformer beat the simple baseline? By how much? Is it worth the cost?


In [ ]:
torch.manual_seed(SEED)
my_model = SensorTransformer(
    NUM_CHANNELS,
    d_model=32,
    nhead=2,
    num_layers=1,
)
my_model = train_sensor_model(my_model, epochs=10, lr=1e-3)

n_params = sum(p.numel() for p in my_model.parameters())
print("This model has", n_params, "parameters.")


### 🧭 Critical reflection: the label is a value judgement

Our model learned to detect "events" — but **we** defined an event as a PM2.5 bump
above the noise. In the real world:

- Who sets that threshold? A government, a company, or the people breathing the air?
- "Normal" readings can still be harmful over long exposure. A model trained to
  flag only spikes can make slow, chronic harm invisible.
- Sensors fail, drift, and are unevenly placed. Wealthier areas often have more
  sensors — so the *data itself* can encode existing inequalities.


## 🧪 More experiments — compare model families and test like it's deployed

A transformer is *one* option, not automatically the best one. And a model that
scores well on clean test data can still fail when a real sensor breaks. The next
experiments build a rival model, ablate a key component, and stress-test robustness.


### 🧪 Experiment 1 — build a 1D-CNN for time series (your own model)

**Why this experiment?** Convolutions slide a small filter across time, so they are
naturally good at spotting *local* patterns — exactly like our PM2.5 bump. CNNs are
cheaper and often match or beat transformers on small sensor datasets. Building one
gives you an honest rival to compare against (and reuses the same training loop).


In [ ]:
class SensorCNN(nn.Module):
    def __init__(self, num_channels, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(num_channels, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.head(x)

torch.manual_seed(SEED)
cnn_model = train_sensor_model(SensorCNN(NUM_CHANNELS), epochs=15, lr=1e-3)
print("Compare this CNN's accuracy to the transformer and the logistic-regression baseline.")


### 🧪 Experiment 2 — does positional encoding even help here?

**Why this experiment?** Positional encoding tells the transformer *when* each
reading happened. But our task is "is there a bump *somewhere* in the window?" —
where the exact position may not matter much. We train with and without it to test
the assumption. Lesson: components that are essential for one task can be irrelevant
for another. Always check, don't assume.


In [ ]:
torch.manual_seed(SEED)
with_pe = train_sensor_model(SensorTransformer(NUM_CHANNELS, use_pos_encoding=True), epochs=12)
print("--- now without positional encoding ---")
torch.manual_seed(SEED)
without_pe = train_sensor_model(SensorTransformer(NUM_CHANNELS, use_pos_encoding=False), epochs=12)
print("Did removing positional encoding hurt? For 'is there a bump anywhere?' it often barely matters.")


### 🧪 Experiment 3 — sensor-failure robustness

**Why this experiment?** Real community sensors clog, drift, and die. A model that
silently depends on one sensor will give confident-but-wrong answers when it fails.
Here we corrupt one channel at a time *at test only* and measure the accuracy drop —
revealing how fragile the model is and which sensor it leans on most.


In [ ]:
from sklearn.metrics import accuracy_score

def evaluate_on(model, X_array, y_array):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_array).to(DEVICE))
    preds = logits.argmax(-1).cpu().numpy()
    return accuracy_score(y_array, preds)

def corrupt_channel(X_array, channel, mode="zero", noise=3.0, seed=0):
    X_bad = X_array.copy()
    if mode == "zero":
        X_bad[:, :, channel] = 0.0
    elif mode == "noise":
        rng = np.random.default_rng(seed)
        X_bad[:, :, channel] += rng.normal(0, noise, X_bad[:, :, channel].shape)
    return X_bad

clean_acc = evaluate_on(model, X_test, y_test)
print(f"Clean test accuracy: {clean_acc:.3f}")
for ch, name in enumerate(CHANNELS):
    broken = evaluate_on(model, corrupt_channel(X_test, ch, mode="zero"), y_test)
    print(f"  if '{name}' sensor flatlines -> accuracy {broken:.3f}  (drop {clean_acc - broken:+.3f})")


### ▶️ Play with it — occlusion: which timesteps does the model rely on?

**Why this experiment?** A score does not tell you *why* a model decided. Here we
hide one timestep at a time and watch how much the "event" probability falls — a
simple, robust way to see where the model looks. For an event window, importance
should spike right where the PM2.5 bump is. This kind of check is how you build
(or withhold) trust before acting on a model's output.


In [ ]:
import matplotlib.pyplot as plt

def event_probability(model, window):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(window[None]).to(DEVICE))
        return logits.softmax(-1)[0, 1].item()

event_window = X_test[np.where(y_test == 1)[0][0]]
base_prob = event_probability(model, event_window)

importance = []
for t in range(SEQ_LEN):
    masked = event_window.copy()
    masked[t, :] = 0.0
    importance.append(base_prob - event_probability(model, masked))

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(range(SEQ_LEN), importance, alpha=0.5, label="importance (drop in P(event))")
ax1.set_xlabel("timestep")
ax1.set_ylabel("importance")
ax2 = ax1.twinx()
ax2.plot(event_window[:, CHANNELS.index("pm25")], color="red", marker=".", label="PM2.5 (standardised)")
ax2.set_ylabel("PM2.5")
plt.title("Which timesteps drive the 'event' decision? (should track the PM2.5 bump)")
fig.legend(loc="upper right")
plt.show()


## 🔴 Optional / Advanced — a Hugging Face time-series model (forecasting)

Hugging Face ships dedicated time-series transformers. Here we use **PatchTST** to
**forecast** the next readings from the recent past — useful for *early warning*
("PM2.5 is about to spike"). Time-series model APIs are newer and evolve, so if an
import fails, run `!pip install -U transformers` and restart the runtime.


In [ ]:
!pip -q install -U transformers

from transformers import PatchTSTConfig, PatchTSTForPrediction

CONTEXT_LENGTH = 32
PREDICTION_LENGTH = 8

long_series, _ = make_sensor_dataset(n_per_class=60, seq_len=CONTEXT_LENGTH + PREDICTION_LENGTH, seed=7)
long_series = (long_series - mean) / std
print("Series windows:", long_series.shape)


In [ ]:
import torch

past = torch.tensor(long_series[:, :CONTEXT_LENGTH, :])
future = torch.tensor(long_series[:, CONTEXT_LENGTH:, :])

config = PatchTSTConfig(
    num_input_channels=NUM_CHANNELS,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    patch_length=8,
    patch_stride=8,
    d_model=64,
    num_attention_heads=4,
    num_hidden_layers=2,
    ffn_dim=128,
    dropout=0.2,
)
ts_model = PatchTSTForPrediction(config).to(DEVICE)
print("PatchTST ready with", sum(p.numel() for p in ts_model.parameters()), "parameters.")


In [ ]:
optimizer = torch.optim.Adam(ts_model.parameters(), lr=1e-3)
past_d, future_d = past.to(DEVICE), future.to(DEVICE)

ts_model.train()
for epoch in range(40):
    optimizer.zero_grad()
    output = ts_model(past_values=past_d, future_values=future_d)
    output.loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:>2}  forecasting loss: {output.loss.item():.4f}")


In [ ]:
import matplotlib.pyplot as plt

ts_model.eval()
with torch.no_grad():
    pred = ts_model(past_values=past_d[:1]).prediction_outputs.cpu().numpy()[0]

pm_index = CHANNELS.index("pm25")
history = past[0, :, pm_index].numpy()
truth = future[0, :, pm_index].numpy()
forecast = pred[:, pm_index]

plt.figure(figsize=(10, 4))
plt.plot(range(CONTEXT_LENGTH), history, label="past", marker=".")
plt.plot(range(CONTEXT_LENGTH, CONTEXT_LENGTH + PREDICTION_LENGTH), truth, label="true future", marker=".")
plt.plot(range(CONTEXT_LENGTH, CONTEXT_LENGTH + PREDICTION_LENGTH), forecast, label="forecast", marker="x")
plt.axvline(CONTEXT_LENGTH - 0.5, color="grey", linestyle="--")
plt.title("PatchTST forecast of PM2.5 (standardised)")
plt.legend()
plt.show()


## 🧭 Closing reflection & exercises

**Reflection**
- Forecasting enables *early warning* — but a wrong forecast can cause false alarms
  or false reassurance. Who is accountable when an alert is wrong?
- Community-run sensing can rebalance power, but only if the community controls the
  **data, the thresholds, and the response**. Otherwise it just feeds someone else's system.

**Exercises**
1. Change how `make_sensor_dataset` defines an "event" (smaller bumps, slow drift).
   Retrain. Which definition is "right"? Who should decide?
2. Make the classes imbalanced (e.g. 700 normal, 80 event). What happens to accuracy
   vs. macro-F1? Why is this realistic for rare pollution events?
3. Add a 4th noisy sensor that carries no real signal. Does the transformer get
   distracted? Does the baseline?
4. Design a one-paragraph **data governance** plan for a real community sensor
   network: who stores the data, who can access it, and who sets the alert thresholds?
